# Projeto de Estatística e Probabilidade
## Seleções de elite no ranking Elo da Copa do Mundo de 2026

**Integrantes:** Victor de Pinho Sampaio e Pedro Augusto Teixeira  
**Dataset:** ratings Elo históricos das 48 seleções classificadas para a Copa do Mundo de 2026  
**Fonte:** Kaggle

Este notebook concentra a evolução principal do trabalho. As células devem ser executadas em ordem. A lógica segue o padrão do projeto anterior da disciplina: importação, carregamento, inspeção, tratamento, qualidade dos dados, análise exploratória, classificação e exportação.

## 1. Instalação das dependências

No terminal do VS Code, execute uma única vez:

```bash
pip install -r requirements.txt
```

O carregamento principal é feito diretamente pelo KaggleHub. O CSV em `data/raw` permanece como cópia local de contingência e rastreabilidade.


## 2. Importação das bibliotecas e configuração do ambiente

Os imports aproveitam o padrão do notebook anterior e acrescentam os recursos necessários para classificação, avaliação e organização do projeto.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from IPython.display import display

try:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter
except ImportError:
    kagglehub = None
    KaggleDatasetAdapter = None

warnings.filterwarnings("ignore")
sb.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

LOCAL_PATH = ROOT / "data" / "raw" / "elo_ratings_wc2026.csv"
CLEAN_PATH = ROOT / "data" / "processed" / "elo_ratings_wc2026_clean.csv"
KAGGLE_DATASET = "afonsofernandescruz/2026-fifa-world-cup-historical-elo-ratings"
KAGGLE_FILE = "elo_ratings_wc2026.csv"

## 3. Carregamento da base

A prioridade é carregar o arquivo diretamente do Kaggle com `kagglehub`. Caso a conexão não esteja disponível, o notebook utiliza a cópia local presente no repositório.


In [ ]:
if kagglehub is not None:
    try:
        df_raw = kagglehub.dataset_load(
            KaggleDatasetAdapter.PANDAS,
            KAGGLE_DATASET,
            KAGGLE_FILE,
        )
        origem = "KaggleHub"
    except Exception:
        df_raw = pd.read_csv(LOCAL_PATH)
        origem = "CSV local de contingência"
else:
    df_raw = pd.read_csv(LOCAL_PATH)
    origem = "CSV local de contingência"

print(f"Origem utilizada: {origem}")
display(df_raw.head())


## 4. Análise exploratória inicial

Nesta etapa, verificamos dimensão, tipos, estatísticas descritivas, duplicatas exatas, valores ausentes e percentuais de ausência.

In [ ]:
print(f"Linhas: {df_raw.shape[0]}")
print(f"Colunas: {df_raw.shape[1]}")

In [ ]:
df_raw.info()

In [ ]:
display(df_raw.describe(include="all").T)

In [ ]:
print(f"Duplicatas exatas: {int(df_raw.duplicated().sum())}")

In [ ]:
display(df_raw.isna().sum().to_frame("nulos"))

In [ ]:
display((df_raw.isna().mean() * 100).sort_values(ascending=False).round(2).to_frame("percentual_nulos"))

## 5. Tratamento, limpeza e engenharia de atributos

As decisões adotadas foram:

- conversão de `snapshot_date` para data;
- validação das somas de partidas e resultados;
- remoção de 48 duplicatas semânticas de 2026, preservando o retrato cronologicamente anterior;
- exclusão de retratos com menos de 30 partidas acumuladas;
- criação de taxas de vitórias, empates e derrotas;
- criação de indicadores de gols por partida e participação por tipo de mando;
- criação da variável categórica `elite_top10`;
- sinalização dos outliers pelo intervalo interquartil, sem exclusão automática.

Não houve imputação de valores, porque a base original não contém campos ausentes.

In [ ]:
from src.data_processing import clean_data, load_raw_data, save_clean_data

df_raw = load_raw_data(LOCAL_PATH)
df, resumo_limpeza = clean_data(df_raw)
save_clean_data(df, CLEAN_PATH)

display(pd.Series(resumo_limpeza, name="resultado"))

In [ ]:
print(f"Dimensão original: {df_raw.shape}")
print(f"Dimensão tratada: {df.shape}")
print(f"Total de nulos restantes: {int(df.isna().sum().sum())}")
print(f"Seleções distintas: {int(df['country'].nunique())}")
display(df.head())

## 6. Tabela de qualidade dos dados após o tratamento

In [ ]:
qualidade = pd.DataFrame({
    "nulos": df.isna().sum(),
    "percentual_nulos": (df.isna().mean() * 100).round(2),
    "tipo": df.dtypes.astype(str),
}).sort_values("percentual_nulos", ascending=False)

display(qualidade)

## 7. Criação de subconjuntos temáticos

Os recortes organizam as análises sem modificar a base tratada:

- `df_desempenho`: indicadores esportivos acumulados;
- `df_confederacoes`: comparação entre confederações;
- `df_tempo`: evolução anual;
- `df_modelagem`: variáveis utilizadas na classificação.

In [ ]:
df_desempenho = df.copy()
df_confederacoes = df.copy()
df_tempo = df.copy()
df_modelagem = df.copy()

print("Base completa:", df.shape)
print("Base de desempenho:", df_desempenho.shape)
print("Base por confederações:", df_confederacoes.shape)
print("Base temporal:", df_tempo.shape)
print("Base para modelagem:", df_modelagem.shape)

## 8. Análises exploratórias principais

A variável-alvo é `elite_top10`:

- `1`: seleção entre as dez melhores posições do ranking Elo no retrato anual;
- `0`: seleção fora do Top 10.

In [ ]:
distribuicao_alvo = df["elite_top10"].map({0: "Fora do Top 10", 1: "Elite (Top 10)"}).value_counts().rename_axis("classe").to_frame("observacoes")
display(distribuicao_alvo)

In [ ]:
resumo_confederacoes = df_confederacoes.groupby("confederation").agg(
    observacoes=("elite_top10", "size"),
    observacoes_elite=("elite_top10", "sum"),
    proporcao_elite=("elite_top10", "mean"),
    rating_medio=("rating_avg", "mean"),
).sort_values("proporcao_elite", ascending=False)

display(resumo_confederacoes)

In [ ]:
colunas_correlacao = [
    "rating_avg",
    "win_rate",
    "draw_rate",
    "loss_rate",
    "goals_for_per_match",
    "goals_against_per_match",
    "goal_diff_per_match",
    "matches_total",
    "elite_top10",
]

correlacoes = df[colunas_correlacao].corr()
display(correlacoes.round(3))

## 9. Visualizações para o dashboard

Cada gráfico possui um objetivo analítico explícito, como exigido na atividade.

In [ ]:
contagem_alvo = df["elite_top10"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(contagem_alvo.index.astype(str), contagem_alvo.values)
plt.title("Distribuição da variável-alvo: elite_top10")
plt.xlabel("Classe: 0 = fora do Top 10; 1 = elite")
plt.ylabel("Quantidade de observações")
plt.show()

In [ ]:
proporcao_confederacoes = resumo_confederacoes["proporcao_elite"].sort_values(ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(proporcao_confederacoes.index, proporcao_confederacoes.values)
plt.title("Proporção histórica de observações no Top 10 por confederação")
plt.xlabel("Confederação")
plt.ylabel("Proporção de elite")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
df.boxplot(column="rating_avg", by="elite_top10")
plt.suptitle("")
plt.title("Rating Elo médio histórico conforme a classe")
plt.xlabel("Classe: 0 = fora do Top 10; 1 = elite")
plt.ylabel("Rating Elo médio histórico")
plt.show()

In [ ]:
rating_mediano_ano = df_tempo.groupby("year")["rating"].median()

plt.figure(figsize=(11, 4))
plt.plot(rating_mediano_ano.index, rating_mediano_ano.values)
plt.title("Evolução anual da mediana do rating Elo")
plt.xlabel("Ano")
plt.ylabel("Mediana do rating Elo")
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sb.heatmap(correlacoes, annot=True, fmt=".2f")
plt.title("Matriz de correlação dos indicadores selecionados")
plt.tight_layout()
plt.show()

In [ ]:
ultimo_ano = int(df["year"].max())
top_15 = df.loc[df["year"] == ultimo_ano, ["country", "rank", "rating", "rating_avg", "confederation", "elite_top10"]].sort_values("rank").head(15)
display(top_15)

## 10. Exportação da base tratada

Esta etapa gera o CSV que será utilizado pelo dashboard e pelos modelos.

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Base tratada exportada para: {CLEAN_PATH}")

## 11. Aplicação manual do Teorema de Bayes

O classificador calcula:

1. probabilidades a priori das classes;
2. verossimilhanças condicionais dos atributos;
3. probabilidades a posteriori normalizadas;
4. suavização de Laplace para evitar probabilidades iguais a zero.

Os atributos numéricos são discretizados em três faixas: baixo, médio e alto.

In [ ]:
from src.bayes_manual import ManualCategoricalNaiveBayes

bayes_numeric_features = ["rating_avg", "win_rate", "goal_diff_per_match", "matches_total"]
bayes_categorical_features = ["confederation"]
bayes_features = bayes_numeric_features + bayes_categorical_features

treino = df.loc[df["year"] <= 2018].copy()
teste = df.loc[df["year"] > 2018].copy()

bayes_model = ManualCategoricalNaiveBayes(
    numeric_features=bayes_numeric_features,
    categorical_features=bayes_categorical_features,
    alpha=1.0,
)
bayes_model.fit(treino[bayes_features], treino["elite_top10"])

priori = pd.Series(bayes_model.priors_, name="probabilidade_a_priori")
display(priori.to_frame())

In [ ]:
perfil_exemplo = teste.loc[teste["country"] == "Brazil", bayes_features].tail(1)
detalhes_bayes, posterior_bayes = bayes_model.explain_prediction(perfil_exemplo)

display(perfil_exemplo)
display(detalhes_bayes)
display(posterior_bayes)

## 12. Regressão logística

A regressão logística utiliza padronização das variáveis numéricas, codificação das variáveis categóricas e balanceamento das classes.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "matches_total",
    "win_rate",
    "draw_rate",
    "goals_for_per_match",
    "goals_against_per_match",
    "goal_diff_per_match",
    "neutral_share",
    "rating_avg",
]
categorical_features = ["confederation", "is_host"]
model_features = numeric_features + categorical_features

logistic_preprocessing = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

logistic_model = Pipeline([
    ("preprocessing", logistic_preprocessing),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])

logistic_model.fit(treino[model_features], treino["elite_top10"])
logistic_prediction = logistic_model.predict(teste[model_features])
logistic_probability = logistic_model.predict_proba(teste[model_features])[:, 1]

## 13. Árvore de decisão e comparação dos classificadores

A árvore de decisão complementa a análise com regras de classificação interpretáveis. Os três métodos são avaliados no recorte temporal de 2019 a 2026.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_preprocessing = ColumnTransformer([
    ("numeric", "passthrough", numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

tree_model = Pipeline([
    ("preprocessing", tree_preprocessing),
    ("classifier", DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, class_weight="balanced", random_state=42)),
])

tree_model.fit(treino[model_features], treino["elite_top10"])
tree_prediction = tree_model.predict(teste[model_features])
tree_probability = tree_model.predict_proba(teste[model_features])[:, 1]
bayes_prediction = bayes_model.predict(teste[bayes_features])
bayes_probability = bayes_model.predict_proba(teste[bayes_features])[:, 1]

In [ ]:
def metricas(nome, real, previsto, probabilidade):
    return {
        "modelo": nome,
        "acuracia": accuracy_score(real, previsto),
        "precisao": precision_score(real, previsto, zero_division=0),
        "recall": recall_score(real, previsto, zero_division=0),
        "f1_score": f1_score(real, previsto, zero_division=0),
        "roc_auc": roc_auc_score(real, probabilidade),
    }

metricas_modelos = pd.DataFrame([
    metricas("Bayes manual", teste["elite_top10"], bayes_prediction, bayes_probability),
    metricas("Regressão logística", teste["elite_top10"], logistic_prediction, logistic_probability),
    metricas("Árvore de decisão", teste["elite_top10"], tree_prediction, tree_probability),
]).sort_values("f1_score", ascending=False)

display(metricas_modelos.round(3))

In [ ]:
matrizes_confusao = {
    "Bayes manual": confusion_matrix(teste["elite_top10"], bayes_prediction),
    "Regressão logística": confusion_matrix(teste["elite_top10"], logistic_prediction),
    "Árvore de decisão": confusion_matrix(teste["elite_top10"], tree_prediction),
}

for nome, matriz in matrizes_confusao.items():
    print(nome)
    display(pd.DataFrame(matriz, index=["real_fora", "real_elite"], columns=["predito_fora", "predito_elite"]))

In [ ]:
metricas_plot = metricas_modelos.set_index("modelo")[["acuracia", "precisao", "recall", "f1_score", "roc_auc"]]

plt.figure(figsize=(10, 5))
metricas_plot.plot(kind="bar", ax=plt.gca())
plt.title("Comparação de desempenho dos três classificadores")
plt.xlabel("Modelo")
plt.ylabel("Resultado")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 14. Conclusões principais

- a base real exigiu tratamento, mesmo sem valores nulos;
- duplicatas semânticas e históricos muito curtos foram tratados de forma justificada;
- o rating Elo médio histórico, a taxa de vitórias e o saldo de gols por partida ajudam a diferenciar a elite;
- a regressão logística obteve a maior acurácia e o maior ROC AUC;
- a árvore de decisão alcançou o maior recall e identificou todas as observações reais de elite no teste;
- o Bayes manual apresentou desempenho competitivo e evidencia o cálculo probabilístico exigido na disciplina.